In [2]:
from transformers import AutoTokenizer, AutoModel

model_dir = "../models/deberta-v3-large-hf-weights"

tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
model = AutoModel.from_pretrained(model_dir, local_files_only=True)

print(model)

/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._py

DebertaV2Model(
  (embeddings): DebertaV2Embeddings(
    (word_embeddings): Embedding(128100, 1024, padding_idx=0)
    (LayerNorm): LayerNorm((1024,), eps=1e-07, elementwise_affine=True)
    (dropout): StableDropout()
  )
  (encoder): DebertaV2Encoder(
    (layer): ModuleList(
      (0-23): 24 x DebertaV2Layer(
        (attention): DebertaV2Attention(
          (self): DisentangledSelfAttention(
            (query_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (key_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (value_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (pos_dropout): StableDropout()
            (dropout): StableDropout()
          )
          (output): DebertaV2SelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-07, elementwise_affine=True)
            (dropout): StableDropout()
          )
        )
        

In [3]:
# Look inside the weights

import torch

state_dict = torch.load(f"{model_dir}/pytorch_model.bin", map_location="cuda:0")
print(type(state_dict))
print(list(state_dict.keys())[:50])

<class 'dict'>
['deberta.embeddings.word_embeddings.weight', 'deberta.embeddings.position_embeddings.weight', 'deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.self.query_proj.weight', 'deberta.encoder.layer.0.attention.self.query_proj.bias', 'deberta.encoder.layer.0.attention.self.key_proj.weight', 'deberta.encoder.layer.0.attention.self.key_proj.bias', 'deberta.encoder.layer.0.attention.self.value_proj.weight', 'deberta.encoder.layer.0.attention.self.value_proj.bias', 'deberta.encoder.layer.0.attention.output.dense.weight', 'deberta.encoder.layer.0.attention.output.dense.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.intermediate.dense.weight', 'deberta.encoder.layer.0.intermediate.dense.bias', 'deberta.encoder.layer.0.output.dense.weight', 'deberta.encoder.layer.0.output.dense.bias', 'deberta.encoder.layer.0.output.Laye

In [5]:
# Check the .pth file

ckpt = torch.load("../models/deberta-large-ls03-ctx1024.pth", map_location="cuda:0")
print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

<class 'collections.OrderedDict'>
odict_keys(['deberta.embeddings.word_embeddings.weight', 'deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.self.query_proj.weight', 'deberta.encoder.layer.0.attention.self.query_proj.bias', 'deberta.encoder.layer.0.attention.self.key_proj.weight', 'deberta.encoder.layer.0.attention.self.key_proj.bias', 'deberta.encoder.layer.0.attention.self.value_proj.weight', 'deberta.encoder.layer.0.attention.self.value_proj.bias', 'deberta.encoder.layer.0.attention.output.dense.weight', 'deberta.encoder.layer.0.attention.output.dense.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.intermediate.dense.weight', 'deberta.encoder.layer.0.intermediate.dense.bias', 'deberta.encoder.layer.0.output.dense.weight', 'deberta.encoder.layer.0.output.dense.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deb

In [6]:
# Check with SentencePiece

import sentencepiece as spm

sp = spm.SentencePieceProcessor()
ok = sp.load(f"{model_dir}/spm.model")
print("Loaded:", ok)

text = "This is a test sentence."
ids = sp.encode(text, out_type=int)
pieces = sp.encode(text, out_type=str)

print("Pieces:", pieces)
print("Ids:", ids)
print("Decoded:", sp.decode(ids))

Loaded: True
Pieces: ['▁This', '▁is', '▁a', '▁test', '▁sentence', '.']
Ids: [329, 269, 266, 1010, 4378, 260]
Decoded: This is a test sentence.


In [7]:
# Run a quick forward pass to get embeddings for a test sentence (uses existing tokenizer, model, text, ids)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

# Tokenize the text and move tensors to device
inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    last_hidden = outputs.last_hidden_state  # (batch, seq_len, hidden)
    cls_emb = last_hidden[:, 0, :]           # use first token embedding as pooled representation
    mean_emb = last_hidden.mean(dim=1)       # mean-pooled representation

print("input tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist()))
print("last_hidden shape:", last_hidden.shape)
print("CLS embedding shape:", cls_emb.shape)
print("Mean pooled shape:", mean_emb.shape)
print("CLS embedding (first 8 values):", cls_emb[0, :8].cpu().numpy())

# Alternative: run model with existing ids list
ids_tensor = torch.tensor([ids], device=device)
with torch.no_grad():
    out_ids = model(input_ids=ids_tensor)
print("Using ids -> last_hidden shape:", out_ids.last_hidden_state.shape)

input tokens: ['[CLS]', '▁This', '▁is', '▁a', '▁test', '▁sentence', '.', '[SEP]']
last_hidden shape: torch.Size([1, 8, 1024])
CLS embedding shape: torch.Size([1, 1024])
Mean pooled shape: torch.Size([1, 1024])
CLS embedding (first 8 values): [-0.3851148  -0.17030415  0.12637945 -0.09393495  0.14636579 -0.35822666
  0.37152943  0.39171842]
Using ids -> last_hidden shape: torch.Size([1, 6, 1024])


In [8]:
# validator-style checks for the loaded model
model.eval()

assert isinstance(model, type(model)), "Loaded object is not a model"
assert hasattr(model, "forward"), "Model must have forward()"
assert hasattr(model, "config"), "Model should have a config"

test_texts = [
    text,
    "Validator test sentence.",
    "The quick brown fox jumps over the lazy dog."
]

for t in test_texts:
    tokenized = tokenizer(t, return_tensors="pt")
    tokenized = {k: v.to(device) for k, v in tokenized.items()}

    with torch.no_grad():
        out = model(**tokenized)

    assert out.last_hidden_state is not None, f"No last_hidden_state for {t!r}"
    assert out.last_hidden_state.shape[0] == 1, "Expected batch size 1"
    assert out.last_hidden_state.shape[1] == tokenized["input_ids"].shape[1]
    assert out.last_hidden_state.shape[2] == model.config.hidden_size

    print(f"PASS: {t!r} -> shape {out.last_hidden_state.shape}")

with torch.no_grad():
    alt_out = model(input_ids=ids_tensor)

assert alt_out.last_hidden_state.shape == ids_tensor.shape + (model.config.hidden_size,)
print("PASS: alternate input_ids path ->", alt_out.last_hidden_state.shape)

PASS: 'This is a test sentence.' -> shape torch.Size([1, 8, 1024])
PASS: 'Validator test sentence.' -> shape torch.Size([1, 7, 1024])
PASS: 'The quick brown fox jumps over the lazy dog.' -> shape torch.Size([1, 12, 1024])
PASS: alternate input_ids path -> torch.Size([1, 6, 1024])


In [9]:
# simple binary classification head for sentence-level AI/human detection
classifier_head = torch.nn.Sequential(
    torch.nn.Dropout(0.1),
    torch.nn.Linear(model.config.hidden_size, 1),
    torch.nn.Sigmoid()
).to(device)

def predict_ai_generated(sentence: str):
    inputs_ = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs_ = model(**inputs_)
        cls_vector = outputs_.last_hidden_state[:, 0, :]
        score = classifier_head(cls_vector).squeeze(-1)
    return score.item(), score.item() >= 0.5

for sample in [
    "This is a test sentence.",
    "The quick brown fox jumps over the lazy dog."
]:
    prob, is_ai = predict_ai_generated(sample)
    print(f"{sample!r} -> ai_prob={prob:.4f}, predicted_ai={is_ai}")

'This is a test sentence.' -> ai_prob=0.4913, predicted_ai=False
'The quick brown fox jumps over the lazy dog.' -> ai_prob=0.4576, predicted_ai=False


In [14]:
predict_ai_generated("You already have a working pipeline there; “running with a test value and getting a result” basically means:")

(0.4723345637321472, False)